---
toc: false
title: Causality Crash Course
date: "06/20/2026"
description: Correlation is not causation, except when it is.
categories: causal_inference
image: /enso-thumbnail.jpg
image-alt: The enso circle
---



Well, hello there, dear reader! 
Today I'm excited to welcome you to the first post about what is likely to be a new recurring theme on this blog — *Causal Inference*.
That's right, it turns out that I've been off learning and practicing causal  for the last few years without mentioning it to you at all.
I meant to write about it a long time ago, but it's been notoriously difficult for me to wrap my keyboard around it.
Hopefully today we can correct this ommission to everyone's satisfaction.

Here's the plan. Since this is the first time we're talking about causality, I'll start with a little pitch section — why should you care and why should you consider bothering to put in the very considerable effort to learn this too.
I'll contextualize causal inference as an area of data science, placing it in the landscape of statistics, machine learning, data analysis, .


Once again it's been a while since I posted here, and we've got a lot to talk about.
Today we're going to begin a new chapter here on *Random Realizations*.

Today I'm going to start catching you up on something I've been off woodshedding for a couple years without mentioning it to you at all; appologies for my silence, but it's taken me a while to wrap my head around it sufficiently to tell you about it.


That's right, today we're talking *Causal Inference*.

## Why causal inference 

### The Purpose of Data Science

Every once in a while my mom asks me if I can re-explain to her what data scientists do. It's a great question really—we in the field should probably ask ourselves more often. In a single sentence I'd say: the purpose of data science is to create value from data.  And what's valuable to the organizations that employ us?  Well an organization has goals, and in order to achieve those goals it needs to make decisions about which actions to take. At its most effective, data science is about decision intelligence—informing a decision making process to help achieve desired outcomes. These decisions come in many forms, from executive deliberation on bold strategic bets to adaptive transaction-level automation occurring millions of times per day.

Examples include:
* What price to charge for this ride share trip?
* Which product to recommend to this user?
* Should a school system reduce its class sizes?
* Where to deploy mobile health centers in an underserved rural area?

Behind each of these decisions is a set of what if questions of the form: What would happen if I did this? What would happen if I did that? To inform the decision we can predict answers to these what if questions. 

Often these answers are already valuable, but we can go one step further and recommend the action corresponding to the most favorable outcome. Thus our business problems can often be formulated as optimization problems of the form

$$ \underset{x}{\operatorname{argmax}} f(x) $$

where $x$ indexes the possible actions we can take and $f$ is a "what if" function that predicts the benefit of an action. The above expression says: "find the action that maximizes the predicted benefit."

So that's the setup; the key for optimal decision making is having a "what if" function that reliably maps actions to outcomes.

For data scientists of my generation who spent the early part of their careers getting good at machine learning, this probably  sounds an awful lot like a prediction problem.
We could follow the typical ML playbook , training a model that minimizes out of sample prediction error.
That means it would be the best predictor of  outcomes from actions  and an ideal objective function to use in our optimization task right?

Well, not necessarily.

My goal for this post is to convince you of why that is. I'll show you why using accuracy-maxing ML models as optimization objectives can fail, not just slightly, but catastrophically. 
To explain why this can happen and how to avoid the scarier pitfalls, we need to talk about *causality*.


## Causality

We live in a causal universe.
The Buddha articulated this 2500 years ago in the Idappaccayatā :

When this is, that is.
From the arising of this comes the arising of that.
When this isn't, that isn't.
From the cessation of this comes the cessation of that.


One event  can cause another event to occur, and often this relationship flows in only one direction.
Cold weather causes people to wear jackets, but wearing jackets does not bring about cold weather.
Of course we have an intuitive grasp of causality from daily life, but in more complicated systems, things can get quite tricky.
So let's build up the mental model we need to reason correctly  about causality  in data science.

We have some system of interest which generates data.
That data comes in the form of observations of some set of variables.
We can model such a system by identifying its data generating process —a simulation that generates observations of the variables of interest.
I think of this as a probabilistic program that assigns a value to each variable  by drawing from some probability distribution whose parameters may or may not depend on the realized values of some of the other variables in the system.
Let's take a look at a simple example.













In [16]:
import numpy as np

rng = np.random.default_rng()
cold = rng.binomial(n=1, p=0.5, size=1)
jacket = rng.binomial(n=1, p=0.1 + 0.8 * cold)
cold, jacket

(array([0]), array([0]))

The above DGP first flips a coin to determine if it's cold or not, then it sets whether the jacket appears with 10% probability  if it's not cold and a 90% probability if it is cold. 
Here, The distribution of cold has nothing to do with jackets, but the distribution of jacket is impacted by the realized value of cold.
We can represent this flow of information with a causal directed acyclic graph.

In this causal DAG, the arrow from cold to jacket indicates that cold causes jacket.

Here comes the key concept to help us understand causality — the *intervention*.
Imagine that we could reach into this system,   set one of the variables however we wanted, then let the other variables respond as they liked.
If we intervene on cold by setting it to cold, then jacket is a Bernoulli draw with $p=0.1%, but if we set it to not cold, then jacket is a Bernoulli draw with $p=0.9$.
Since interventions on cold change the probability distribution of jacket, we can say that cold causes jacket.
On the other hand, no matter how we intervene on jacket, cold is still a Bernoulli draw with $p=0.5$.
Since interventions on jacket have no effect on cold, jacket does not cause cold.

This may seem trite, but intervention is the way to rigourously understand causal relationships.


##  Causality and the Pitfalls of Prediction

Let's gently introduce some of the key ideas in causal modeling and highlight potential pitfalls of using prediction for decision making  by walking through a concrete example.


Suppose there's an e-commerce firm that's interested in using promotional offers to increase customer retention.
They've got a historical customer-level dataset that tells whether each customer was given an offer and whether they were retained.
The data science team plans to build a model to predict customer retention from promo status and then use that to decide whether to give out another round of promos.
Here's how the system works:

* Customers have various prior risks of churning, and risk has a strong negative effect on retention.
* Promos have a small positive effect on customer retention. 
* If we remain passive, then the sales team tends to offer promos to customers they think might be at risk of churning.
* If we decide to intervene, we can step in and override whether or not customers receive promo offers, regardless of what the sales team would have done on their own.


Let's think of this system in terms of its data generating process.

Let's build a little simulation that can generate data from this system.


In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

def simulate_data(n=10000, promo_override=None):
    # 1. Simulate risk (continuous variable)
    risk = np.random.normal(0, 1, n)

    # 2. Simulate promo based on risk or set to specific value
    if promo_override is None:
        promo_prob = 1 / (1 + np.exp(-1.5 * risk))
        promo = np.random.binomial(1, promo_prob)
    else:
        promo = np.full(n, promo_override) if np.isscalar(promo_override) else np.array(promo_override)

    # 3. Simulate retention (higher risk less likely to retain, promo helps slightly)
    retention_logit = 1.0 - 2.0 * risk + 0.5 * promo
    retention_prob = 1 / (1 + np.exp(-retention_logit))
    retention = np.random.binomial(1, retention_prob)

    return pd.DataFrame({
        'customer_id': np.arange(1, n + 1),
        # 'risk': risk,
        'promo': promo,
        'retention': retention
    })

df = simulate_data(n=1000)
df.head()

,customer_id,promo,retention
0,1,1,1
1,2,0,1
2,3,1,1
3,4,1,0
4,5,0,1


Notice that by default (`promo_override = None`), all three variables, `risk`, `promo`, and `retention`, are random.
But interestingly, the distribution of `promo` depends on the realized value of `risk` — higher `risk`, higher probability of `promo`.
Similarly, the realized values of `risk` and `promo` both affect the probability of `retention`.
We can understand causality as the situation where the realized value of one variable influences the distribution of another variable.



In [6]:
#| code-fold: true
mean_retention = df.groupby('promo')['retention'].mean()
print(f"Average retention without promo: {mean_retention[0]:.2%}")
print(f"Average retention with promo: {mean_retention[1]:.2%}")

Average retention without promo: 81.09%
Average retention with promo: 56.06%


When the DS team checked the retention rates for customers with and without promo offers, they were surprised to find that retention was actually. lower for those who had received the promos.
They mused: "that's odd; our promos must be pretty terrible or something."
Then they proceeded intrepidly, building a model to predict retention using promo status.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Train-test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Features and target
X_train = train_df[['promo']]
y_train = train_df['retention']
X_test = test_df[['promo']]
y_test = test_df['retention']

# Fit a Random Forest model on train data
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

# Evaluate on holdout set
holdout_predictions = clf.predict(X_test)
accuracy = accuracy_score(y_test, holdout_predictions)
print(f"Holdout accuracy: {accuracy:.2%}")

# Predict retention for with and without promo
promo_scenarios = pd.DataFrame({'promo': [0, 1]})
predictions = clf.predict_proba(promo_scenarios)[:, 1]

print(f"Predicted retention without promo: {predictions[0]:.2%}")
print(f"Predicted retention with promo: {predictions[1]:.2%}")

# Choose the optimal action (argmax)
best_promo_idx = np.argmax(predictions)
best_promo_val = promo_scenarios.loc[best_promo_idx, 'promo']
print(f"Optimal decision: promo = {best_promo_val}")

Holdout accuracy: 70.40%
Predicted retention without promo: 81.05%
Predicted retention with promo: 57.79%
Optimal decision: promo = 0


 Using their model, they found that  `promo = 0`, no promotional offer, gives the highest retention prediction and is therefore the  optimal decision for the firm.

Womp womp.

---

So what went wrong?
It turns out that our intrepid DS team has fallen victim to the age old saying — 

> *correlation does not imply causation*.

Predictive models are very good at answering  *observational* questions — questions about what sorts of patterns have been typically observed in the past.
Specifically, supervised ML models find the *observational* conditional expectation function:

$$ \hat{f}(x) = E[Y|X=x] $$

which tells us what the response $Y$ tends to look like in training data when feature $X$ happens to take value $x$. In other words, it lives in the realm of correlations.
The DS team's model learned this observational conditional expectation function, which showed that when promos were applied, retention tended to be low; this much was correct.


But when our ill-fated DS team tried to use a predictive model to answer a what if question about promos and their effect on retention, they were actually asking an *interventional* question — a question about what happens when we decide to intervene on a system in a particular way.
The function they actually wanted to know is the *interventional* conditional expectation function:

$$ \hat{f}(x) = E[Y|do(X=x)] $$

which tells us what the response $Y$ tends to look like when some intervention, represented by the *do operator*,  forcibly sets $X$ to value $x$.
This is what the DS team actually wanted — a function that gives the expected retention when promo is set rather than observed.
This expresses a very different situation from the observational distribution.
Consider how promo arises in the observational data; because the sales team was targeting customers with high churn risk, promo tends to occur in cases with low retention, even though it has a slight positive effect on retention.
In contrast, when promo arises from an intervention, promo is no longer associated with high churn risk, and thus we're able to see its positive effect on retention.

So how do we estimate this interventional expectation function?
In general the observational and interventional conditional expectation functions are not the same, and they are only the same when certain special conditions are satisfied.
It  is the function of causal inference  to figure out how to satisfy those conditions so that we can use observational data to answer causal questions.
In other words:

> "Correlation does not imply causation — except when it does.*"


They had assumed that their model would correctly pick up the *causal effect* of promo on retention, but in fact, all it did was capture the *correlation* between them.





## References 

* [Causal Inference for the Brave and True](https://matheusfacure.github.io/python-causality-handbook/landing-page.html)
* Robert's Course 
* Book of Why
* Statistical Rethinking
* Robert's book